<h2 align="center">Codebasics ML Course: SMS Spam Classification Using Naive Bayes</h2>

### Dataset Credits
This dataset is taken from UCI Machine Learning Respository: https://archive.ics.uci.edu/dataset/228/sms+spam+collection

Refer to dataset_credits_readme.txt file for credits details.

Naive Bayes Rule Explained
To understand Naive Bayes, we first need to understand Bayes’ Theorem, which updates a belief (probability of an event) based on new evidence.

Let:

A be the event: "Person is COVID positive"

B be the evidence: "Person has flu-X symptoms"

1. Posterior Probability:
P(A|B) = Probability that a person is COVID positive given they have flu-X.

This is what we want to calculate (our updated belief after seeing evidence B).

2. Likelihood:
P(B|A) = Probability of having flu-X if the person is COVID positive.

This tells us how likely the evidence is, assuming A is true.

3. Prior Probability:
P(A) = Our belief in someone being COVID positive before seeing the flu-X symptom.

4. Marginal Probability:
P(B) = Total probability of someone having flu-X regardless of COVID status.

Bayes' Theorem Formula:
 P(A|B) = P(B|A) * P(A) / P(B)
 


### Data Load and EDA

In [1]:
import pandas as pd

df = pd.read_csv("spam.csv")
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [2]:
df['Category'].value_counts()

Category
ham     4825
spam     747
Name: count, dtype: int64

### Train Test Split

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.Message,df.Category, test_size=0.2, random_state=5)

In [4]:
X_train[2]

"Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's"

### Feature Engineering: Convert Text into Vector

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

v = CountVectorizer()
X_train_count = v.fit_transform(X_train.values)
X_test_count = v.transform(X_test.values)

X_train_count.toarray()[:2]

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(2, 7705))

In [6]:
len(X_train_count.toarray()[1])

7705

### Train a Model Using Naive Bayes

In [7]:
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()
model.fit(X_train_count,y_train)

MultinomialNB()

### Model Evaluation: Precision, Recall and F1-Score

In [8]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test_count)

report = classification_report(y_test, y_pred)
print(report)

              precision    recall  f1-score   support

         ham       0.99      1.00      1.00       970
        spam       0.98      0.96      0.97       145

    accuracy                           0.99      1115
   macro avg       0.99      0.98      0.98      1115
weighted avg       0.99      0.99      0.99      1115



### Model Evaluation: Confusion Matrix

In [10]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
from matplotlib import pyplot as plt

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", xticklabels=['Not Spam', 'Spam'], yticklabels=['Not Spam', 'Spam'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix Visualization')
plt.show()

ModuleNotFoundError: No module named 'seaborn'

### Prediction on Sample Emails

In [47]:
emails = [
    'Hey mohan, can we get together to watch footbal game tomorrow?',
    'Upto 20% discount on parking, exclusive offer just for you. Dont miss this reward!'
]
emails_count = v.transform(emails)
model.predict(emails_count)

array(['ham', 'spam'], dtype='<U4')

### Now do same thing using Sklearn Pipeline with Few Lines of Code

In [48]:
from sklearn.pipeline import Pipeline
pipe = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('nb', MultinomialNB())
])

pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)
report = classification_report(y_test, y_pred)
print(report)

              precision    recall  f1-score   support

         ham       0.99      1.00      1.00       970
        spam       0.98      0.96      0.97       145

    accuracy                           0.99      1115
   macro avg       0.99      0.98      0.98      1115
weighted avg       0.99      0.99      0.99      1115



In [50]:
pipe.predict(emails)

array(['ham', 'spam'], dtype='<U4')

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Sample documents
documents = [
    "I love programming in Python",
    "Python programming is fun",
    "I love coding and programming"
]

# Initialize TfidfVectorizer
vectorizer = TfidfVectorizer()

# Learn vocabulary and transform documents into TF-IDF feature matrix
tfidf_matrix = vectorizer.fit_transform(documents)

# Get feature names (words)
feature_names = vectorizer.get_feature_names_out()

# Convert sparse matrix to dense to see actual values
dense_matrix = tfidf_matrix.todense()

# Show feature names and TF-IDF vectors
print("Features (words):", feature_names)
print("\nTF-IDF matrix:")
for i, doc_vector in enumerate(dense_matrix):
    print(f"Document {i+1}:", doc_vector)


Features (words): ['and' 'coding' 'fun' 'in' 'is' 'love' 'programming' 'python']

TF-IDF matrix:
Document 1: [[0.         0.         0.         0.63174505 0.         0.4804584
  0.37311881 0.4804584 ]]
Document 2: [[0.         0.         0.5844829  0.         0.5844829  0.
  0.34520502 0.44451431]]
Document 3: [[0.5844829  0.5844829  0.         0.         0.         0.44451431
  0.34520502 0.        ]]
